# 👑 GeoMAS - Research Master Dashboard

Questo notebook è progettato per supportare l'analisi dei dati per la Tesi Magistrale.
È diviso in un capitolo di **Telemetria Standard** (per monitorare la salute della simulazione) e quattro capitoli dedicati alle **Research Questions (RQ)**.

### Indice dei Capitoli:
- **Capitolo 0**: Telemetria Standard (Preserved Legacy)
- **Capitolo 1 (RQ1)**: Scenario Comparison (Baseline vs Variant)
- **Capitolo 2 (RQ2)**: Etica e Tensioni Decisionali
- **Capitolo 3 (RQ3)**: Onestà Strategica e Moral Washing
- **Capitolo 4 (RQ4)**: Emergenza di Equilibri Socio-Politici

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.lines as mlines
import numpy as np

# Configurazione Stile Master
sns.set_theme(style="darkgrid", context="talk")
plt.rcParams['figure.figsize'] = (14, 7)
COLORS = {'defense': '#e74c3c', 'foreign': '#3498db', 'economy': '#f1c40f', 'satisfaction': '#2ecc71'}

DB_PATH = 'simulation_metrics.duckdb'

def query_db(query, params=None):
    if params:
        params = [p.item() if hasattr(p, 'item') else p for p in params]
    with duckdb.connect(DB_PATH, read_only=True) as conn:
        return conn.execute(query, params or []).df()

## ⚙️ Selezione Dati
Scegli la simulazione principale (Target) e, opzionalmente, una simulazione di confronto (Baseline).

In [ ]:
simulations = query_db("SELECT DISTINCT simulation_id FROM metrics_global ORDER BY simulation_id")
print("Simulazioni Disponibili:")
display(simulations)

# TARGET: L'ultima simulazione (quella su cui stiamo lavorando)
TARGET_ID = simulations['simulation_id'].iloc[-1] if not simulations.empty else None

# BASELINE: La penultima (o una specifica) per il confronto RQ1
BASELINE_ID = simulations['simulation_id'].iloc[-2] if len(simulations) > 1 else None

print(f"Target Sim: {TARGET_ID}")
print(f"Baseline Sim: {BASELINE_ID} (per RQ1 Comparison)")

def load_sim_data(sim_id):
    g = query_db("SELECT * FROM metrics_global WHERE simulation_id = ? ORDER BY turn", [sim_id])
    n = query_db("SELECT * FROM metrics_nation WHERE simulation_id = ? ORDER BY turn, nation_id", [sim_id])
    t = query_db("SELECT * FROM metrics_trust WHERE simulation_id = ? ORDER BY turn", [sim_id])
    return g, n, t

global_df, nation_df, trust_df = load_sim_data(TARGET_ID)

## 📊 Capitolo 0: Telemetria Standard (Legacy)
Monitoraggio delle metriche base di tutte le nazioni nella simulazione Target.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 18))

# 0.1 performance medie (Deception & Coherence)
sns.lineplot(data=global_df, x='turn', y='global_deception_avg', ax=axes[0], label='Deception Media', color='red')
sns.lineplot(data=global_df, x='turn', y='global_coherence_avg', ax=axes[0], label='Coherence Media', color='green')
axes[0].set_title("Performance Agenti (Global Avg)")
axes[0].set_ylim(-0.1, 1.1)

# 0.2 Guns vs Butter
ax_butter = axes[1]
ax_guns = ax_butter.twinx()
sns.lineplot(data=global_df, x='turn', y='global_trade_volume', ax=ax_butter, color='blue', label='Trade')
sns.lineplot(data=global_df, x='turn', y='units_created', ax=ax_guns, color='orange', label='Military Units')
ax_butter.set_title("Guns vs Butter (Global Allocations)")

# 0.3 Power Projection & Satisfaction
sns.lineplot(data=nation_df, x='turn', y='power_projection', hue='nation_id', ax=axes[2])
axes[2].set_title("Evoluzione del Potere (Power Projection per Nazione)")

plt.tight_layout()
plt.show()

## 🔍 Capitolo 1 (RQ1): Scenario Comparison
Confronto tra la simulazione corrente (Target) e la Baseline selezionata.

In [ ]:
if BASELINE_ID:
    g_base, n_base, t_base = load_sim_data(BASELINE_ID)
    
    comparison_df = pd.concat([
        global_df.assign(sim_type='Target'),
        g_base.assign(sim_type='Baseline')
    ])
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    sns.lineplot(data=comparison_df, x='turn', y='global_deception_avg', hue='sim_type', ax=axes[0])
    axes[0].set_title("Delta Deception (Target vs Baseline)")
    
    sns.lineplot(data=comparison_df, x='turn', y='global_trade_volume', hue='sim_type', ax=axes[1])
    axes[1].set_title("Delta Trade Volume (Target vs Baseline)")
    
    plt.show()
else:
    print("Necessaria una seconda simulazione per il confronto.")

## 🛡️ Capitolo 3 (RQ3): Onestà e Moral Washing
Analisi della dicotomia tra l'onestà diplomatica (pubblica) e militare (privata).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 3.1 The Deception Gap (Moral Washing Index)
gap_df = nation_df.groupby('turn')[['deception_foreign', 'deception_defense']].mean().reset_index()
gap_df['gap'] = gap_df['deception_foreign'] - gap_df['deception_defense']
axes[0].fill_between(gap_df['turn'], gap_df['deception_defense'], gap_df['deception_foreign'], color='purple', alpha=0.2, label='Deception Gap')
axes[0].plot(gap_df['turn'], gap_df['deception_defense'], label='Defense (Private)', color=COLORS['defense'], marker='x')
axes[0].plot(gap_df['turn'], gap_df['deception_foreign'], label='Foreign (Public)', color=COLORS['foreign'], marker='o')
axes[0].set_title("Deception Gap: Differenza tra Onestà Pubblica e Privata")
axes[0].legend()

# 3.2 Deception per Nazione
sns.barplot(data=nation_df[nation_df['turn'] == nation_df['turn'].max()], x='nation_id', y='deception_overall', ax=axes[1], palette='flare')
axes[1].set_title("Ranking Deception Score (Ultimo Turno)")

plt.show()

## 🌐 Capitolo 4 (RQ4): Equilibri e Convergenza
Analisi del network per determinare se il sistema si stabilizza o collassa.

In [ ]:
plt.figure(figsize=(10, 8))
last_turn = nation_df['turn'].max()
t_last = trust_df[trust_df['turn'] == last_turn]

if not t_last.empty:
    G = nx.DiGraph()
    REL_MAP = {'MUTUAL_DEFENSE': 'solid', 'NON_AGGRESSION': 'dashed', 'WAR': 'solid', 'PEACE': 'dotted'}
    COLOR_MAP = {'MUTUAL_DEFENSE': 'green', 'NON_AGGRESSION': 'lime', 'WAR': 'red', 'PEACE': 'gray'}
    
    for _, row in t_last.iterrows():
        if row['observer_id'] == row['target_id']: continue
        G.add_edge(row['observer_id'], row['target_id'], color=COLOR_MAP.get(row['relationship_state'], 'gray'), style=REL_MAP.get(row['relationship_state'], 'dotted'))

    pos = nx.spring_layout(G, seed=42)
    colors = [G[u][v]['color'] for u, v in G.edges()]
    nx.draw(G, pos, with_labels=True, node_color='lightgray', node_size=3000, edge_color=colors, connectionstyle='arc3,rad=0.1', width=2)
    plt.title(f"Stato delle Relazioni Globali (Turno {last_turn})")
    plt.show()

# 4.2 Stability Index
stability_df = nation_df.groupby('turn')['public_satisfaction'].mean().reset_index()
plt.figure(figsize=(14, 5))
sns.lineplot(data=stability_df, x='turn', y='public_satisfaction', color='green', linewidth=4)
plt.title("Stability Index (Media Soddisfazione Mondiale)")
plt.axhline(50, color='gray', linestyle='--')
plt.show()